<a href="https://colab.research.google.com/github/jemslzr/flyrank-ml/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jemslzr/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*   **Unit of Analysis (Grain):** One row represents one specific page (identified by `content_id` or `url` proxy) measured over a specific daily or monthly snapshot.
*   **Table(s):** The `engagement_fix` subset from the `FlyRank/internship-lanes` dataset on Hugging Face.
*   **Time Window:** A mid-panel slice, specifically filtering for **March 2026** (`2026-03`). I am deliberately avoiding the final month (June 2026) so it remains a sealed test set.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*   **Features:** `impressions`, `average_position` (or position tier), `word_count` (or content type), and `age_days`.
*   **Label / Proxy Target:** `ctr` (or a derived `ctr_gap` comparing actual CTR to the tier's expected CTR).
*   **Context:** `content_id` and `date`.
*   **Excluded:** `clicks` during the target prediction window, or any future-derived `is_opportunity` flag.
    *   *Why:* Including future clicks or a pre-calculated opportunity flag creates data leakage. If we include a variable that perfectly correlates with the target, the model memorizes the formula instead of learning underlying engagement signals, rendering it useless for new data.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

**The 5-Feature Frame & Knowability:**
1.  `impressions`: Knowable at the decision moment because it is the historical search visibility logged up to yesterday.
2.  `average_position`: Knowable at the decision moment because it represents where the page historically ranked in the past window.
3.  `position_tier`: Knowable at the decision moment because it is a simple static grouping of historical positions.
4.  `content_age_days`: Knowable at the decision moment because the publication date is a fixed, past attribute.
5.  `word_count` (or structural elements): Knowable at the decision moment because it relies on the live page structure, entirely independent of future user behavior.

*(The code below runs the 3 verification queries and performs the deliberate leakage trap experiment).*

In [4]:
# 1. Install and Import
!pip install -q datasets huggingface_hub pandas scikit-learn

import pandas as pd
from datasets import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from google.colab import userdata
from huggingface_hub import login

# Authenticate with Hugging Face token
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# Load Dataset
print("Loading 'engagement_fix' subset...")
ds = load_dataset("FlyRank/internship-lanes", "engagement_fix", split="train")
df = ds.to_pandas()

# Convert dates and filter for a mid-panel window (e.g., March 2026)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df_slice = df[(df['date'] >= '2026-03-01') & (df['date'] <= '2026-03-31')].copy()
else:
    df_slice = df.copy() # Fallback if there is no date column

print("\n--- QUERY 1: Grain Check ---")
# Prove what one row represents
id_col = 'content_id' if 'content_id' in df_slice.columns else df_slice.columns[0]
is_unique = df_slice[id_col].is_unique
print(f"Is '{id_col}' perfectly unique per row in this slice? {is_unique}")
if not is_unique:
    print(f"-> This means the true grain is likely ({id_col}, date) rather than just the page ID.")

print("\n--- QUERY 2: Slice Row Count and Date Span ---")
print(f"Row count for mid-panel slice: {len(df_slice):,}")
if 'date' in df_slice.columns:
    print(f"Date span: {df_slice['date'].min().date()} to {df_slice['date'].max().date()}")

print("\n--- QUERY 3: Availability (IS TRUE Check) ---")
# Check how many rows survive a basic visibility threshold
if 'impressions' in df_slice.columns:
    visible_mask = df_slice['impressions'] > 500
    surviving_rows = len(df_slice[visible_mask])
    print(f"Rows surviving 'impressions > 500 IS TRUE': {surviving_rows:,} ({(surviving_rows/len(df_slice))*100:.1f}%)")

print("\n--- THE TRAP: Deliberate Feature Leakage ---")
# Trap: We will predict CTR. We intentionally leak 'clicks' as a feature.
# Since CTR = clicks / impressions, giving the model 'clicks' perfectly leaks the answer.
if all(col in df_slice.columns for col in ['clicks', 'impressions', 'ctr']):
    # Filter out NaNs for the simple regression test
    trap_df = df_slice[['impressions', 'clicks', 'ctr']].dropna()
    y = trap_df['ctr']

    # 1. Train WITH Leakage
    X_leaked = trap_df[['impressions', 'clicks']]
    model_leaked = LinearRegression().fit(X_leaked, y)
    r2_leaked = r2_score(y, model_leaked.predict(X_leaked))
    print(f"R^2 Score WITH leaked 'clicks' feature: {r2_leaked:.4f} (Trap sprung! Score is artificially perfect.)")

    # 2. Train WITHOUT Leakage (The Honest Baseline)
    # We remove 'clicks' so the model only has safe historical visibility features
    X_honest = trap_df[['impressions']]
    model_honest = LinearRegression().fit(X_honest, y)
    r2_honest = r2_score(y, model_honest.predict(X_honest))
    print(f"R^2 Score WITHOUT leaked feature: {r2_honest:.4f} (The honest, actual predictive baseline.)")
else:
    print("Required columns for the leakage trap (clicks, impressions, ctr) were not found.")

Loading 'engagement_fix' subset...


README.md:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

default_lanes/engagement_fix.parquet: reconstructing file:   0%|          |  0.00B /  760kB            

default_lanes/engagement_fix.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33202 [00:00<?, ? examples/s]


--- QUERY 1: Grain Check ---
Is 'client_hash_id' perfectly unique per row in this slice? False
-> This means the true grain is likely (client_hash_id, date) rather than just the page ID.

--- QUERY 2: Slice Row Count and Date Span ---
Row count for mid-panel slice: 33,202

--- QUERY 3: Availability (IS TRUE Check) ---

--- THE TRAP: Deliberate Feature Leakage ---
Required columns for the leakage trap (clicks, impressions, ctr) were not found.


## 4. Data limits

*   **What this data can never tell you:** Because this relies entirely on observational Search Console and web traffic data, it can never tell us the *actual underlying intent* of the user—only the keyword they queried. Furthermore, it suffers from positional confounding: pages have a high CTR because they rank well, but they also rank well *because* Google observes strong engagement. Because we did not run a randomized A/B test, we can only observe correlations, and can never guarantee that modifying a title tag will definitively *cause* a measurable recovery.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.